# 第 11 天：动量因子 2

> 来自《30 天因子研究计划》第 11 天  
> 主题：动量因子 2  
> 必做：RSI  
> 选做：MACD  
> 目标产出：技术因子库

---

## 0. 今天你要真正学会什么？

第 10 天的动量因子直接使用过去收益。  
今天学习更常见的技术指标：RSI 和 MACD。

它们的核心问题是：


价格强弱和趋势变化，是否能被转化成稳定信号？


今天掌握：

1. RSI 如何衡量上涨和下跌力量。
2. MACD 如何用快慢均线捕捉趋势变化。
3. 如何把 RSI、MACD 转成可检验的因子库。
4. 为什么技术指标也必须经过 IC 和回测检验。

---

## 1. RSI 的直觉

RSI 是相对强弱指标。它比较一段时间内上涨幅度和下跌幅度：


RSI = 100 - 100 / (1 + RS)
RS = 平均上涨 / 平均下跌


常见解释：

- RSI 高：近期上涨力量强。
- RSI 低：近期下跌力量强。
- RSI 过高可能过热。
- RSI 过低可能超跌。

---

## 2. MACD 的直觉

MACD 使用快慢指数移动均线：


DIF = EMA(short) - EMA(long)
DEA = EMA(DIF)
MACD histogram = DIF - DEA


直觉：

- 快线高于慢线：短期趋势强于长期趋势。
- 柱状图上升：趋势加速。
- 柱状图下降：趋势减弱。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260706)


---

## 4. 构造模拟价格数据


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=260)
tickers = [f"Stock_{i:03d}" for i in range(80)]

price = pd.DataFrame(index=dates, columns=tickers, dtype=float)
for i, ticker in enumerate(tickers):
    drift = rng.normal(0.00025, 0.0002)
    cyc = 0.002 * np.sin(np.linspace(0, 8 * np.pi, len(dates)) + i / 7)
    ret = drift + cyc + rng.normal(0, 0.017, len(dates))
    price[ticker] = 60 * np.cumprod(1 + ret)

price.head()


---

## 5. 计算 RSI


In [ ]:
def rsi(close: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    diff = close.diff()
    gain = diff.clip(lower=0)
    loss = -diff.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)


rsi_14 = rsi(price, 14)
rsi_14.tail()


RSI 通常在 0 到 100 之间。作为因子时，可以直接用，也可以做中心化：


In [ ]:
rsi_factor = (rsi_14 - 50) / 50
rsi_factor.tail()


---

## 6. 计算 MACD


In [ ]:
def macd(close: pd.DataFrame, fast: int = 12, slow: int = 26, signal: int = 9) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    hist = dif - dea
    return dif, dea, hist


dif, dea, macd_hist = macd(price)
macd_hist.tail()


为了不同价格股票可比，可以除以价格：


In [ ]:
macd_factor = macd_hist / price
macd_factor.tail()


---

## 7. 生成技术因子库


In [ ]:
def wide_to_long(wide: pd.DataFrame, name: str) -> pd.DataFrame:
    return (
        wide.stack(future_stack=True)
        .dropna()
        .rename(name)
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "ticker"})
    )


tech_library = wide_to_long(rsi_factor, "rsi_14_factor")
tech_library = tech_library.merge(wide_to_long(macd_factor, "macd_hist_factor"), on=["date", "ticker"], how="outer")
tech_library.head()


---

## 8. 快速检验 IC


In [ ]:
future_10d_ret = price.shift(-10) / price - 1
label = wide_to_long(future_10d_ret, "future_10d_ret")
data = tech_library.merge(label, on=["date", "ticker"], how="inner").dropna()

ic_rows = {}
for col in ["rsi_14_factor", "macd_hist_factor"]:
    daily_ic = data.groupby("date").apply(
        lambda g: g[col].corr(g["future_10d_ret"], method="spearman"),
        include_groups=False
    )
    ic_rows[col] = {
        "mean_rank_ic": daily_ic.mean(),
        "icir": daily_ic.mean() / daily_ic.std(),
        "positive_ratio": (daily_ic > 0).mean(),
    }

pd.DataFrame(ic_rows).T


---

## 9. 目标产出：技术因子库函数


In [ ]:
def build_technical_factor_library(close: pd.DataFrame) -> pd.DataFrame:
    rsi_14 = (rsi(close, 14) - 50) / 50
    _, _, hist = macd(close)
    macd_hist_factor = hist / close

    out = wide_to_long(rsi_14, "rsi_14_factor")
    out = out.merge(wide_to_long(macd_hist_factor, "macd_hist_factor"), on=["date", "ticker"], how="outer")
    return out


technical_library = build_technical_factor_library(price)
technical_library.tail()


---

## 10. 知识图谱


In [ ]:
mindmap
  root((动量因子2))
    RSI
      上涨力量
      下跌力量
      超买超卖
    MACD
      快慢均线
      DIF
      DEA
      柱状图
    因子化
      中心化
      价格归一
      IC检验
      技术因子库


---

## 11. 作业

1. 把 RSI 窗口从 14 改成 6 和 24。
2. 把 MACD 参数改成 6/19/9，比较 IC。
3. 思考 RSI 高是追趋势还是反转信号。
4. 用第 5 天分层回测检验 RSI 因子。

---

## 12. 自测题

1. RSI 主要比较什么？  
   答案：一段时间内平均上涨和平均下跌力量。

2. MACD 的 DIF 是什么？  
   答案：快 EMA 减慢 EMA。

3. 技术指标能直接实盘吗？  
   答案：不能，仍需要 IC、分层回测和交易成本检验。

---

## 13. 明天预告

明天学习波动率因子：历史波动率和 ATR。

---

## 14. 仅供学习的提醒

本文使用模拟数据解释 RSI 和 MACD 构建方法，不构成任何投资建议。

---

# 统一高质量增强模块

> 本增强模块用于把第 11 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：动量因子2
- 必做：RSI
- 选做：MACD
- 目标产出：技术因子库

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

RSI 像观察多空力量的温度计，MACD 像观察趋势加速度的仪表盘。技术指标要变成因子，就必须可量化、可排序、可检验。

这个例子背后的关键直觉是：

> 技术指标不是神秘图形，而是价格序列的函数。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


动量因子2
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 技术因子库


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(111)
dates = pd.bdate_range("2024-01-02", periods=180)
close = pd.Series(100 * np.cumprod(1 + rng.normal(.0004, .016, len(dates))), index=dates)
diff = close.diff()
gain = diff.clip(lower=0).rolling(14).mean()
loss = (-diff.clip(upper=0)).rolling(14).mean()
rsi = 100 - 100 / (1 + gain / loss)
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
dif = ema12 - ema26
dea = dif.ewm(span=9, adjust=False).mean()
macd_hist = dif - dea
result = pd.DataFrame({"close": close, "rsi14": rsi, "macd_hist_pct": macd_hist / close})
print(result.tail().round(4))


## E. 产出验收标准

完成今天课程后，你的 `技术因子库` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `动量因子2` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `技术因子库` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `技术因子库`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 11 天复盘：动量因子2

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
---

# 第 10-15 天深度加厚模块


    ## L. 为什么还要加厚这一课？

    这一课属于第 10-15 天的“工程化因子”部分：它不像 Alpha、Beta 那样只靠概念就能建立直觉，也不像 PE、ROE 那样有明确财务含义。它更依赖窗口、参数、预处理顺序和检验口径。

    所以学习 `动量因子2` 时，不能只停留在“知道公式”。你至少要完成三层理解：


    第一层：公式能写对
    第二层：参数变化后结果还能解释
    第三层：能放进统一因子流水线


    如果只学第一层，代码很快能写出来，但研究时很容易陷入“参数换一下结果就变”的困境。

    ## M. 更贴近真实研究的场景

    RSI 和 MACD 的问题不是公式难，而是解释难。同一个 RSI，高位可能代表强趋势，也可能代表过热；低位可能代表超跌，也可能代表弱者恒弱。

    这类问题在真实研究中很常见：一个因子看似简单，但只要换股票池、换窗口、换持有期、换市场阶段，结果就会明显变化。成熟的研究方式不是逃避这种变化，而是把变化记录下来、解释出来。

    ## N. 参数敏感性实验

    下面这段代码是专门为本课补充的参数实验。它不追求复杂，而是训练一个习惯：

    > 不要只交一个因子结果，至少比较几组合理参数。


In [ ]:
    import numpy as np
import pandas as pd

rng = np.random.default_rng(211)
dates = pd.bdate_range("2024-01-02", periods=220)
close = pd.Series(100*np.cumprod(1+rng.normal(0.0002,0.018,len(dates))), index=dates)

def calc_rsi(close, window):
    diff = close.diff()
    gain = diff.clip(lower=0).rolling(window).mean()
    loss = (-diff.clip(upper=0)).rolling(window).mean()
    return 100 - 100/(1 + gain/loss)

def calc_macd(close, fast, slow, signal=9):
    dif = close.ewm(span=fast, adjust=False).mean() - close.ewm(span=slow, adjust=False).mean()
    dea = dif.ewm(span=signal, adjust=False).mean()
    return (dif - dea) / close

result = pd.DataFrame({
    "rsi_6": calc_rsi(close, 6),
    "rsi_14": calc_rsi(close, 14),
    "rsi_24": calc_rsi(close, 24),
    "macd_12_26": calc_macd(close, 12, 26),
    "macd_6_19": calc_macd(close, 6, 19),
})
print(result.tail().round(4))
print(result.corr().round(3))


    ## O. 结果该怎么写进研究笔记？

    建议你用下面这个格式记录：


    因子名称：动量因子2

    1. 使用的数据：
       - 股票池：
       - 时间区间：
       - 价格 / 财务口径：

    2. 核心参数：
       - 主参数：
       - 对照参数：

    3. 因子方向：
       - 因子值越大代表：
       - 是否需要取负号：

    4. 检验结果：
       - Rank IC：
       - ICIR：
       - 分组收益：
       - 多空表现：

    5. 稳定性：
       - 参数变化后是否稳定：
       - 分阶段是否稳定：
       - 极端行情是否失效：

    6. 结论：
       - 是否进入因子库：
       - 还需要什么后续验证：


    ## P. 额外验收清单

    `技术因子库` 如果要达到可复用标准，额外检查：

    1. RSI 至少比较 2 个窗口。
2. MACD 参数需要说明 fast、slow、signal。
3. 技术指标要做价格归一或中心化。
4. 说明高 RSI 是动量逻辑还是反转逻辑。

    ## Q. 更深入的常见误区

    ### 误区 1：参数越多越高级

    参数多不代表研究深，很多时候只是过拟合空间更大。真正高级的是解释参数为什么合理，并证明它在相邻参数下仍然不崩。

    ### 误区 2：只看全样本平均

    全样本平均可能掩盖阶段失效。至少要分年度、分市场状态、分股票池看一次。

    ### 误区 3：把预处理当成机械步骤

    去极值、标准化、中性化会改变因子含义。每加一步，都要知道自己剥离了什么，也可能损失了什么。

    ### 误区 4：忽略交易可行性

    技术、波动率、流动性类因子往往换手更高，交易成本可能非常关键。纸面有效不等于可交易。

    ## R. 加厚作业

    1. 把本课主参数上下各调整一次，记录结果变化。
    2. 把未来收益标签从 20 日改成 5 日和 60 日，观察结论是否变。
    3. 随机删除 10% 股票样本，检查结果是否稳定。
    4. 把因子取反，确认分组结果是否镜像变化。
    5. 写一段 200 字研究结论，必须同时包含“支持证据”和“风险提示”。

    ## S. 一句话升级结论

    `动量因子2` 的高质量学习标准不是“会算”，而是：

    > 会定义、会检验、会解释参数变化，也知道它在真实交易里可能被什么击穿。
